# Experiment 2 — CSSNet + CLAHE preprocessing

Open in **VS Code** (Jupyter + Google Colaboratory). Select Kernel → **Colab** → **T4 GPU**.

On My Drive: `cv_dataset.zip` only. This notebook clones the code from GitHub. Checkpoints go to `MyDrive/cssnet_runs/exp2`.


Same model, seed and hyperparameters as Experiment 1; the only change is the `preprocess` block in `config_exp2.yaml` (CLAHE).

In [1]:
import torch
assert torch.cuda.is_available(), "Select Kernel → Colab → T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")


Tesla T4 14.6 GB


In [7]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_RUNS = "/content/drive/MyDrive/cssnet_runs/exp2"
GIT_URL = "https://github.com/Areesha-008/Abdominal-CT-Segmentation.git"

_mydrive = Path("/content/drive/MyDrive")
_candidates = [
    _mydrive / "cv_dataset.zip",
    _mydrive / "cv_dataset.tar.gz",
    _mydrive / "cv_dataset.tgz",
    _mydrive / "cv_dataset.rar",
    _mydrive / "cv_dataset",
]
DRIVE_DATA = next((p for p in _candidates if p.exists()), None)
assert DRIVE_DATA is not None, "Upload cv_dataset.zip to My Drive (must contain train/ and test/)"
print("data archive:", DRIVE_DATA)


MessageError: Failed to issue request POST https://colab.pa.googleapis.com/v1/credential-propagation:enable: Bad Request
Response body: {
  "error": {
    "code": 400,
    "message": "Precondition check failed.",
    "status": "FAILED_PRECONDITION"
  }
}


In [ ]:
from pathlib import Path
import shutil

CODE = Path("/content/Abdominal-CT-Segmentation")
DATA = Path("/content/Synapse")
UNPACK = Path("/content/_cv_dataset_unpack")

if CODE.exists():
    shutil.rmtree(CODE)
!git clone --depth 1 {GIT_URL} {CODE}
%cd {CODE}

def find_split(root: Path, name: str, pattern: str):
    hits = [p.parent for p in root.rglob(pattern) if p.parent.name == name]
    return hits[0] if hits else None

if not (DATA / "train").is_dir():
    src = Path(DRIVE_DATA)
    if UNPACK.exists():
        shutil.rmtree(UNPACK)
    UNPACK.mkdir(parents=True)

    if src.is_file() and src.suffix == ".zip":
        !unzip -q -o "{src}" -d "{UNPACK}"
    elif src.is_file() and (src.suffixes[-2:] == [".tar", ".gz"] or src.suffix == ".tgz"):
        !tar -xzf "{src}" -C "{UNPACK}"
    elif src.is_file() and src.suffix == ".rar":
        !apt-get -qq install -y unrar
        !unrar x -o+ "{src}" "{UNPACK}/"
    elif src.is_dir():
        !cp -a "{src}" "{UNPACK}/cv_dataset"
    else:
        raise FileNotFoundError(src)

    train_src = find_split(UNPACK, "train", "*.npz")
    test_src = find_split(UNPACK, "test", "*.h5")
    assert train_src and test_src, f"archive must contain train/*.npz and test/*.h5; top level: {list(UNPACK.iterdir())[:20]}"
    DATA.mkdir(parents=True, exist_ok=True)
    shutil.move(str(train_src), str(DATA / "train"))
    shutil.move(str(test_src), str(DATA / "test"))
    shutil.rmtree(UNPACK, ignore_errors=True)

print("train slices", len(list((DATA / "train").glob("*.npz"))))
print("test volumes", len(list((DATA / "test").glob("*.h5"))))


In [ ]:
%pip install -q scipy h5py opencv-python-headless timm ml-collections medpy SimpleITK pyyaml


In [ ]:
from pathlib import Path
import os, sys, subprocess

CODE = Path("/content/Abdominal-CT-Segmentation")
RUN = Path("/content/drive/MyDrive/cssnet_runs/exp2")
DATA = Path("/content/Synapse")
os.chdir(CODE)

cmd = [sys.executable, "train.py", "--config", "config_exp2.yaml", "--out", str(RUN), "--data_root", str(DATA)]
if (RUN / "last.pth").exists():
    cmd.append("--resume")
    print("resume")
else:
    print("fresh run")
subprocess.check_call(cmd)


In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir("/content/Abdominal-CT-Segmentation")
run = Path("/content/drive/MyDrive/cssnet_runs/exp2")
alt = Path("/content/Abdominal-CT-Segmentation") / "{DRIVE_RUNS}"
if not (run / "best.pth").exists() and not (run / "last.pth").exists():
    if (alt / "best.pth").exists() or (alt / "last.pth").exists():
        run = alt
print("evaluate", run, "pth:", list(run.glob("*.pth")))
subprocess.check_call([
    sys.executable, "evaluate.py",
    "--config", "config_exp2.yaml",
    "--run", str(run),
    "--data_root", "/content/Synapse",
])


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

run = Path("/content/drive/MyDrive/cssnet_runs/exp2")
if not (run / "results.json").exists():
    run = Path("/content/Abdominal-CT-Segmentation") / "{DRIVE_RUNS}"
r = json.load(open(run / "results.json"))
DRIVE_RUNS = str(run)
print(f"{'organ':14s} {'Dice':>8s} {'IoU':>8s}")
for o, v in r["per_organ"].items():
    print(f"{o:14s} {v['dice']:8.4f} {v['iou']:8.4f}")
print(f"{'MEAN':14s} {r['mean_dice']:8.4f} {r['mean_iou']:8.4f}")

for p in sorted((Path(DRIVE_RUNS) / "overlays").glob("*.png")):
    display(Image(filename=str(p)))
